# ENS Challenge 60 - Stock Auction Volume Prediction
## Main Pipeline Notebook

**Goal**: Predict the natural logarithm of closing auction volume (as a fraction of total daily volume) for 900 stocks.

**Metric**: RMSE (Root Mean Squared Error)

**Benchmark**: 0.4742 RMSE

## 1. Setup and Imports

In [3]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print('Standard imports loaded!')

Standard imports loaded!


In [4]:
# Import custom utilities
import sys
sys.path.insert(0, '.')

from utils.data_loader import (
    load_config, load_data, merge_target, get_feature_columns,
    create_submission, print_data_info
)
from utils.preprocessing import preprocess_data, handle_missing_values
from utils.feature_engineering import (
    create_features, create_aggregation_features, create_domain_features,
    create_nlv_features, create_day_features, get_feature_list
)
from utils.validation import (
    TimeSeriesValidator, compute_metrics, cross_validate,
    create_temporal_split, print_cv_summary
)
from utils.models import (
    TwoStageModel, train_lightgbm, train_linear,
    get_model, train_and_evaluate
)
from utils.mlflow_utils import setup_mlflow, log_experiment, ExperimentTracker
from utils.visualization import (
    plot_feature_importance, plot_residuals, plot_predictions,
    plot_cv_results, plot_target_distribution, create_summary_dashboard
)

print('Custom utilities loaded!')

Custom utilities loaded!


In [5]:
# Load configuration
config = load_config('configs/config.yaml')
print('Configuration loaded!')
print(f'Train input: {config["data"]["train_input"]}')
print(f'Train output: {config["data"]["train_output"]}')
print(f'Test input: {config["data"]["test_input"]}')

Configuration loaded!
Train input: data/input_training.csv.gz
Train output: data/output_training_IxKGwDV.csv
Test input: data/input_test.csv.gz


## 2. Data Loading

In [6]:
# Load data with memory optimization
X_train_raw, y_train, X_test_raw = load_data(config=config, load_test=True, verbose=True)

Loading training input data...
  Shape: (684482, 127)
  Memory: 347.72 MB
Loading training output data...
  Shape: (684482, 2)
Loading test input data...
  Shape: (311744, 127)
  Memory: 158.37 MB


In [ ]:
# Print data info
print_data_info(X_train_raw, 'Training Data', config)
print_data_info(X_test_raw, 'Test Data', config)

In [ ]:
# Check target distribution
print('Target Statistics:')
print(y_train.describe())

In [ ]:
# Merge target with training data
train_df = merge_target(X_train_raw, y_train, config)
print(f'Merged training data shape: {train_df.shape}')
print(f'Target column: {config["features"]["target_col"]}')

## 3. Data Exploration

In [ ]:
# Check for missing values
feature_cols = get_feature_columns(config)
return_cols = feature_cols['return_cols']
volume_cols = feature_cols['volume_cols']

print('Missing values in returns:', train_df[return_cols].isnull().sum().sum())
print('Missing values in volumes:', train_df[volume_cols].isnull().sum().sum())
print('Missing values in LS:', train_df['LS'].isnull().sum())
print('Missing values in NLV:', train_df['NLV'].isnull().sum())

In [ ]:
# Plot target distribution
target_col = config['features']['target_col']
fig = plot_target_distribution(train_df[target_col].values, title='Target Distribution')
plt.show()

In [ ]:
# Check day range
day_col = config['features']['day_col']
print(f'Training days: {train_df[day_col].min()} to {train_df[day_col].max()}')
print(f'Number of unique days: {train_df[day_col].nunique()}')
print(f'Number of unique stocks: {train_df[config["features"]["id_col"]].nunique()}')

## 4. Preprocessing

In [ ]:
# Preprocess training data
X_train_processed, _, X_test_processed = preprocess_data(
    X_train_raw, y_train, X_test_raw, config=config, verbose=True
)

In [ ]:
# Merge target again after preprocessing
train_df = merge_target(X_train_processed, y_train, config)
print(f'Processed training data shape: {train_df.shape}')

## 5. Feature Engineering

In [ ]:
# Create features for training data
train_df = create_features(
    train_df,
    config=config,
    include_aggregations=True,
    include_domain=True,
    include_nlv=True,
    include_day=True,
    include_temporal=False,
    verbose=True
)

In [ ]:
# Create features for test data
X_test_processed = create_features(
    X_test_processed,
    config=config,
    include_aggregations=True,
    include_domain=True,
    include_nlv=True,
    include_day=True,
    include_temporal=False,
    verbose=True
)

In [ ]:
# Get feature list for modeling (include pid as categorical feature)
feature_list, categorical_cols = get_feature_list(train_df, config, exclude_raw=False, exclude_target=True, include_id=True)
print(f'Total features for modeling: {len(feature_list)}')
print(f'Categorical features: {categorical_cols}')
print(f'Feature list (first 30): {feature_list[:30]}')

## 6. Train/Validation Split

In [ ]:
# Create temporal split
max_day = train_df[day_col].max()
train_end_day = int(max_day * 0.85)

print(f'Max day: {max_day}')
print(f'Train end day: {train_end_day}')

train_split, val_split, _ = create_temporal_split(
    train_df, train_end_day=train_end_day, config=config
)

print(f'Training samples: {len(train_split):,}')
print(f'Validation samples: {len(val_split):,}')

In [ ]:
# Prepare features and target
target_col = config['features']['target_col']

X_train = train_split[feature_list]
y_train_split = train_split[target_col]

X_val = val_split[feature_list]
y_val = val_split[target_col]

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')

## 7. Model Training

### 7.1 Baseline: Linear Regression on NLV

In [ ]:
from sklearn.linear_model import Ridge

nlv_model = Ridge(alpha=1.0)
nlv_model.fit(X_train[['NLV']], y_train_split)

nlv_train_pred = nlv_model.predict(X_train[['NLV']])
nlv_val_pred = nlv_model.predict(X_val[['NLV']])

nlv_train_metrics = compute_metrics(y_train_split.values, nlv_train_pred)
nlv_val_metrics = compute_metrics(y_val.values, nlv_val_pred)

print('=' * 50)
print('BASELINE: Linear Regression on NLV only')
print('=' * 50)
print(f'Train RMSE: {nlv_train_metrics["rmse"]:.4f}')
print(f'Val RMSE:   {nlv_val_metrics["rmse"]:.4f}')
print(f'Val R2:     {nlv_val_metrics["r2"]:.4f}')

### 7.2 Two-Stage Model (CFM Benchmark)

In [ ]:
two_stage_model = TwoStageModel(
    linear_alpha=1.0,
    lgb_params=config['models']['lightgbm']['params'],
    categorical_features=categorical_cols if categorical_cols else None,
    early_stopping_rounds=50,
    config=config
)

two_stage_model.fit(X_train, y_train_split, X_val=X_val, y_val=y_val)

ts_train_pred = two_stage_model.predict(X_train)
ts_val_pred = two_stage_model.predict(X_val)

ts_train_metrics = compute_metrics(y_train_split.values, ts_train_pred)
ts_val_metrics = compute_metrics(y_val.values, ts_val_pred)

print('=' * 50)
print('TWO-STAGE MODEL: Linear + LightGBM')
print('=' * 50)
print(f'Train RMSE: {ts_train_metrics["rmse"]:.4f}')
print(f'Val RMSE:   {ts_val_metrics["rmse"]:.4f}')
print(f'Val R2:     {ts_val_metrics["r2"]:.4f}')

### 7.3 LightGBM Only

In [ ]:
import lightgbm as lgb

lgb_params = config['models']['lightgbm']['params'].copy()
lgb_model = lgb.LGBMRegressor(**lgb_params)

# Use categorical_cols from get_feature_list (or empty list if none)
cat_features = categorical_cols if categorical_cols else 'auto'

lgb_model.fit(
    X_train, y_train_split,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

lgb_train_pred = lgb_model.predict(X_train)
lgb_val_pred = lgb_model.predict(X_val)

lgb_train_metrics = compute_metrics(y_train_split.values, lgb_train_pred)
lgb_val_metrics = compute_metrics(y_val.values, lgb_val_pred)

print('=' * 50)
print('LIGHTGBM ONLY')
print('=' * 50)
print(f'Train RMSE: {lgb_train_metrics["rmse"]:.4f}')
print(f'Val RMSE:   {lgb_val_metrics["rmse"]:.4f}')
print(f'Val R2:     {lgb_val_metrics["r2"]:.4f}')

## 8. Cross-Validation

In [ ]:
print('=' * 50)
print('TIME SERIES CROSS-VALIDATION')
print('=' * 50)

X_full = train_df[feature_list]
y_full = train_df[target_col]

validator = TimeSeriesValidator(n_splits=5, config=config)

lgb_cv_model = lgb.LGBMRegressor(**lgb_params)
cv_results = cross_validate(
    model=lgb_cv_model,
    X=X_full,
    y=y_full,
    validator=validator,
    config=config,
    verbose=True
)

In [ ]:
print_cv_summary(cv_results)

In [ ]:
fig = plot_cv_results(cv_results, title='LightGBM Cross-Validation Results')
plt.show()

## 9. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_list,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 20 Most Important Features:')
print(importance_df.head(20).to_string(index=False))

In [ ]:
fig = plot_feature_importance(importance_df, top_n=25, title='LightGBM Feature Importance')
plt.show()

## 10. Model Analysis

In [ ]:
fig = plot_predictions(y_val.values, lgb_val_pred, title='LightGBM: Predictions vs Actual')
plt.show()

In [ ]:
fig = plot_residuals(y_val.values, lgb_val_pred, title='LightGBM: Residual Analysis')
plt.show()

In [ ]:
fig = create_summary_dashboard(
    y_true=y_val.values,
    y_pred=lgb_val_pred,
    cv_results=cv_results,
    feature_importance=importance_df
)
plt.show()

## 11. Final Model and Submission

In [ ]:
print('Training final model on all training data...')

final_model = lgb.LGBMRegressor(**lgb_params)
final_model.fit(X_full, y_full, categorical_feature=cat_features)

print('Final model trained!')

In [ ]:
print('Making predictions on test data...')

X_test_final = X_test_processed[feature_list]
print(f'Test data shape: {X_test_final.shape}')

test_predictions = final_model.predict(X_test_final)
print(f'Predictions shape: {test_predictions.shape}')
print(f'Predictions range: [{test_predictions.min():.4f}, {test_predictions.max():.4f}]')

In [ ]:
submission = create_submission(
    predictions=test_predictions,
    X_test=X_test_final,
    config=config,
    output_path='outputs/submission_phase1.csv'
)

print('Submission preview:')
print(submission.head(10))
print(f'Submission shape: {submission.shape}')

## 12. Results Summary

In [ ]:
print('=' * 60)
print('PHASE 1 RESULTS SUMMARY')
print('=' * 60)
print(f'Benchmark RMSE: 0.4742')
print(f'Model Performance (Validation Set):')
print(f'  1. NLV Only (Baseline):     RMSE = {nlv_val_metrics["rmse"]:.4f}')
print(f'  2. Two-Stage (Linear+LGB):  RMSE = {ts_val_metrics["rmse"]:.4f}')
print(f'  3. LightGBM Full:           RMSE = {lgb_val_metrics["rmse"]:.4f}')
print(f'Cross-Validation (5-fold TimeSeriesSplit):')
print(f'  LightGBM CV RMSE: {cv_results["mean_rmse"]:.4f} +/- {cv_results["std_rmse"]:.4f}')
print('=' * 60)